# **Aula 1 - Preparando os Dados**

## **1.1 Apresenta??o**

## **1.2 Preparando o Ambiente**

### PySpark

PySpark ? uma interface para Apache Spark em Python. Ele n?o apenas permite que voc? escreva aplicativos Spark usando APIs Python, mas tamb?m fornece o *shell* PySpark para analisar interativamente seus dados em um ambiente distribu?do. O PySpark oferece suporte ? maioria dos recursos do Spark, como Spark SQL, DataFrame, Streaming, MLlib (Machine Learning) e Spark Core.

<center><img src="https://caelum-online-public.s3.amazonaws.com/2273-introducao-spark/01/img-001.png"/></center>

#### **Spark SQL e DataFrame**

Spark SQL ? um m?dulo Spark para processamento de dados estruturados. Ele fornece uma abstra??o de programa??o chamada DataFrame e tamb?m pode atuar como mecanismo de consulta SQL distribu?do.

#### **Spark Streaming**

Executando em cima do Spark, o recurso de *streaming* no Apache Spark possibilita o uso de poderosas aplica??es interativas e anal?ticas em *streaming* e dados hist?ricos, enquanto herda a facilidade de uso do Spark e as caracter?sticas de toler?ncia a falhas.

#### **Spark MLlib**

Constru?do sobre o Spark, MLlib ? uma biblioteca de aprendizado de m?quina escalon?vel que fornece um conjunto uniforme de APIs de alto n?vel que ajudam os usu?rios a criar e ajustar *pipelines* de aprendizado de m?quina pr?ticos.

#### **Spark Core**

Spark Core ? o mecanismo de execu??o geral subjacente para a plataforma Spark sobre o qual todas as outras funcionalidades s?o constru?das. Ele fornece um RDD (*Resilient Distributed Dataset*) e recursos de computa??o na mem?ria.

<font size=2>**Fonte:** [PySpark](https://spark.apache.org/docs/latest/api/python/index.html)</font>

In [ ]:
# !pip install pyspark

### SparkSession

O ponto de entrada para programar o Spark com a API Dataset e DataFrame.

Uma SparkSession pode ser utilizada para criar DataFrames, registrar DataFrames como tabelas, executar consultas SQL em tabelas, armazenar em cache e ler arquivos parquet. Para criar uma SparkSession, use o seguinte padr?o de construtor:

<font size=2>**Fonte:** [SparkSession](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.sql.SparkSession.html)</font>

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.master('local[*]').appName("Classifica??o com Spark").getOrCreate()
spark

## **1.3 Carregamento dos Dados**

In [ ]:
dados = spark.read.csv('dados/dados_clientes.csv', sep=',', header=True, inferSchema=True)

In [ ]:
dados

In [ ]:
dados.show()

In [ ]:
dados.count()

## Target

In [ ]:
dados.select('Churn').groupBy('Churn').count().show()

In [ ]:
dados.printSchema()

## **1.4 Transformando os Dados**

<font size=2>**Fonte:** [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql.html#functions)</font>

In [ ]:
colunasBinarias = [
    'Churn',
    'Conjuge',
    'Dependentes',
    'TelefoneFixo',
    'MaisDeUmaLinhaTelefonica',
    'SegurancaOnline',
    'BackupOnline',
    'SeguroDispositivo',
    'SuporteTecnico',
    'TVaCabo',
    'StreamingFilmes',
    'ContaCorreio'
]

In [ ]:
from pyspark.sql import functions as f

### Preparando os dados

Transformando os dados em 0 e 1 

In [ ]:
todasColunas = [f.when(f.col(c)=='Sim', 1).otherwise(0).alias(c) for c in colunasBinarias]
for coluna in reversed(dados.columns):
  if coluna not in colunasBinarias:
    todasColunas.insert(0, coluna)
todasColunas

In [ ]:
dados.select(todasColunas).show()

In [ ]:
dataset = dados.select(todasColunas)

In [ ]:
dataset.show()

## **1.5 Criando *Dummies***

In [ ]:
dataset.printSchema()

In [ ]:
dataset.select(['Internet', 'TipoContrato', 'MetodoPagamento']).show()

In [ ]:
# coloca 1 quando tem o valor e 0 quando n?o tem 
dataset.groupBy('id').pivot('MetodoPagamento').agg(f.lit(1)).na.fill(0).show()

In [ ]:
Internet = dataset.groupBy('id').pivot('Internet').agg(f.lit(1)).na.fill(0)
TipoContrato = dataset.groupBy('id').pivot('TipoContrato').agg(f.lit(1)).na.fill(0)
MetodoPagamento = dataset.groupBy('id').pivot('MetodoPagamento').agg(f.lit(1)).na.fill(0)

In [ ]:
# juntando os dados - renomeando as colunas - removendo colunas 
dataset\
    .join(Internet, 'id', how='inner')\
    .join(TipoContrato, 'id', how='inner')\
    .join(MetodoPagamento, 'id', how='inner')\
    .select(
        '*',
        f.col('DSL').alias('Internet_DSL'),
        f.col('FibraOptica').alias('Internet_FibraOptica'),
        f.col('Nao').alias('Internet_Nao'),
        f.col('Mensalmente').alias('TipoContrato_Mensalmente'),
        f.col('UmAno').alias('TipoContrato_UmAno'),
        f.col('DoisAnos').alias('TipoContrato_DoisAnos'),
        f.col('DebitoEmConta').alias('MetodoPagamento_DebitoEmConta'),
        f.col('CartaoCredito').alias('MetodoPagamento_CartaoCredito'),
        f.col('BoletoEletronico').alias('MetodoPagamento_BoletoEletronico'),
        f.col('Boleto').alias('MetodoPagamento_Boleto')
    ).drop(
        'Internet', 'TipoContrato', 'MetodoPagamento', 'DSL',
        'FibraOptica', 'Nao', 'Mensalmente', 'UmAno', 'DoisAnos',
        'DebitoEmConta', 'CartaoCredito', 'BoletoEletronico', 'Boleto'
    ).show()

In [ ]:
# salvando a altera??o no dataset
dataset = dataset\
    .join(Internet, 'id', how='inner')\
    .join(TipoContrato, 'id', how='inner')\
    .join(MetodoPagamento, 'id', how='inner')\
    .select(
        '*',
        f.col('DSL').alias('Internet_DSL'),
        f.col('FibraOptica').alias('Internet_FibraOptica'),
        f.col('Nao').alias('Internet_Nao'),
        f.col('Mensalmente').alias('TipoContrato_Mensalmente'),
        f.col('UmAno').alias('TipoContrato_UmAno'),
        f.col('DoisAnos').alias('TipoContrato_DoisAnos'),
        f.col('DebitoEmConta').alias('MetodoPagamento_DebitoEmConta'),
        f.col('CartaoCredito').alias('MetodoPagamento_CartaoCredito'),
        f.col('BoletoEletronico').alias('MetodoPagamento_BoletoEletronico'),
        f.col('Boleto').alias('MetodoPagamento_Boleto')
    ).drop(
        'Internet', 'TipoContrato', 'MetodoPagamento', 'DSL',
        'FibraOptica', 'Nao', 'Mensalmente', 'UmAno', 'DoisAnos',
        'DebitoEmConta', 'CartaoCredito', 'BoletoEletronico', 'Boleto'
    )

In [ ]:
dataset.show()

# Preparando os dados para o modelo

In [ ]:
from pyspark.ml.feature import VectorAssembler

In [ ]:
dataset = dataset.withColumnRenamed('Churn','label')

In [ ]:
x = dataset.columns
x.remove('label')
x.remove('id')

In [ ]:
assembler = VectorAssembler(inputCols=x, outputCol='features')
dataset_prep = assembler.transform(dataset).select('features','label')

In [ ]:
dataset_prep.show(10, truncate=False)

# 24 quantidade features
# representa??o do vetor com coluna que n?o s?o 0
# valores da lista 

In [ ]:
seed = 1

# Treino e Teste

In [ ]:
treino, teste = dataset_prep.randomSplit([0.7,0.3], seed=seed)

In [ ]:
from pyspark.ml.classification import LogisticRegression

In [ ]:
lr = LogisticRegression()
modelo_lr = lr.fit(treino)

In [ ]:
prev = modelo_lr.transform(teste)

In [ ]:
prev.show()

In [ ]:
resume = modelo_lr.summary

In [ ]:
resume.accuracy

In [ ]:
print("Acur?cia: %f" % resume.accuracy)
print("Precis?o: %f" % resume.precisionByLabel[1])
print("Recall: %f" % resume.recallByLabel[1])
print("F1: %f" % resume.fMeasureByLabel()[1])

In [ ]:
prev.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 1)).count()

In [ ]:
tp = prev.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 1)).count()
tn = prev.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 0)).count()
fp = prev.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 1)).count()
fn = prev.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 0)).count()
print(tp, tn,fp, fn)

In [ ]:
def calcula_mostra_matriz_confusao(df_transform_modelo, normalize=False, percentage=True):
  tp = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 1)).count()
  tn = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 0)).count()
  fp = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 0) & (f.col('prediction') == 1)).count()
  fn = df_transform_modelo.select('label', 'prediction').where((f.col('label') == 1) & (f.col('prediction') == 0)).count()
  
  valorP = 1
  valorN = 1

  if normalize:
    valorP = tp + fn
    valorN = fp + tn
  
  if percentage and normalize:
    valorP = valorP / 100
    valorN = valorN / 100

  print(' '*20, 'Previsto')
  print(' '*15, 'Churn', ' '*5 ,'N?o-Churn')
  print(' '*4, 'Churn', ' '*6, int(tp/valorP), ' '*7, int(fn/valorP))
  print('Real')
  print(' '*4, 'N?o-Churn', ' '*2, int(fp/valorN), ' '*7, int(tn/valorN))

In [ ]:
calcula_mostra_matriz_confusao(prev, normalize=False)

# Modelo

# Decision Tree Classifier


Colocar um dataframe o resultado

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

In [ ]:
dtc = DecisionTreeClassifier(seed=seed)
dtc_modelo = dtc.fit(treino)

In [ ]:
prev_dtc = dtc_modelo.transform(treino)

In [ ]:
prev_dtc.show()

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
evaluator = MulticlassClassificationEvaluator()

In [ ]:
evaluator.evaluate(prev_dtc,{evaluator.metricName: "accuracy"})

In [ ]:
print(f"Acur?cia: {round(evaluator.evaluate(prev_dtc, {evaluator.metricName: 'accuracy'}), 4)}")
print(f"Precis?o: {round(evaluator.evaluate(prev_dtc, {evaluator.metricName: 'precisionByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"Recall: {round(evaluator.evaluate(prev_dtc, {evaluator.metricName: 'recallByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"F1: {round(evaluator.evaluate(prev_dtc,  {evaluator.metricName: 'fMeasureByLabel', evaluator.metricLabel: 1}), 4)}")


In [ ]:
val_dtc = dtc_modelo.transform(teste)

In [ ]:
evaluator.evaluate(val_dtc,{evaluator.metricName: "accuracy"})

In [ ]:
print('Decision Tree Classifier')
print("="*40)
print("Dados de Treino")
print("="*40)
print("Matriz de Confus?o")
print("-"*40)
calcula_mostra_matriz_confusao(prev_dtc, normalize=False)
print("-"*40)
print("M?tricas")
print("-"*40)
print(f"Acur?cia: {round(evaluator.evaluate(prev_dtc, {evaluator.metricName: 'accuracy'}), 4)}")
print(f"Precis?o: {round(evaluator.evaluate(prev_dtc, {evaluator.metricName: 'precisionByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"Recall: {round(evaluator.evaluate(prev_dtc, {evaluator.metricName: 'recallByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"F1: {round(evaluator.evaluate(prev_dtc,  {evaluator.metricName: 'fMeasureByLabel', evaluator.metricLabel: 1}), 4)}")
print("")
print("="*40)
print("Dados de Teste")
print("="*40)
print("Matriz de Confus?o")
print("-"*40)
calcula_mostra_matriz_confusao(val_dtc, normalize=False)
print("-"*40)
print("M?tricas")
print("-"*40)
print(f"Acur?cia: {round(evaluator.evaluate(val_dtc, {evaluator.metricName: 'accuracy'}), 4)}")
print(f"Precis?o: {round(evaluator.evaluate(val_dtc, {evaluator.metricName: 'precisionByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"Recall: {round(evaluator.evaluate(val_dtc, {evaluator.metricName: 'recallByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"F1: {round(evaluator.evaluate(val_dtc,  {evaluator.metricName: 'fMeasureByLabel', evaluator.metricLabel: 1}), 4)}")

# Random Florest

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

In [ ]:
rfc = RandomForestClassifier(seed=seed)

In [ ]:
rfc_modelo = rfc.fit(treino)

In [ ]:
rfc_treino = rfc_modelo.transform(treino)

In [ ]:
print(f"Acur?cia: {round(evaluator.evaluate(rfc_treino, {evaluator.metricName: 'accuracy'}), 4)}")

In [ ]:
rfc_teste = rfc_modelo.transform(teste)

In [ ]:
print('Random Florest')
print("="*40)
print("Dados de Treino")
print("="*40)
print("Matriz de Confus?o")
print("-"*40)
calcula_mostra_matriz_confusao(rfc_treino, normalize=False)
print("-"*40)
print("M?tricas")
print("-"*40)
print(f"Acur?cia: {round(evaluator.evaluate(rfc_treino, {evaluator.metricName: 'accuracy'}), 4)}")
print(f"Precis?o: {round(evaluator.evaluate(rfc_treino, {evaluator.metricName: 'precisionByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"Recall: {round(evaluator.evaluate(rfc_treino, {evaluator.metricName: 'recallByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"F1: {round(evaluator.evaluate(rfc_treino,  {evaluator.metricName: 'fMeasureByLabel', evaluator.metricLabel: 1}), 4)}")
print("")
print("="*40)
print("Dados de Teste")
print("="*40)
print("Matriz de Confus?o")
print("-"*40)
calcula_mostra_matriz_confusao(val_dtc, normalize=False)
print("-"*40)
print("M?tricas")
print("-"*40)
print(f"Acur?cia: {round(evaluator.evaluate(rfc_teste, {evaluator.metricName: 'accuracy'}), 4)}")
print(f"Precis?o: {round(evaluator.evaluate(rfc_teste, {evaluator.metricName: 'precisionByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"Recall: {round(evaluator.evaluate(rfc_teste, {evaluator.metricName: 'recallByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"F1: {round(evaluator.evaluate(rfc_teste,  {evaluator.metricName: 'fMeasureByLabel', evaluator.metricLabel: 1}), 4)}")

# Cross Validation

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
grid = ParamGridBuilder()\
        .addGrid(dtc.maxDepth,[2,5,10])\
        .addGrid(dtc.maxBins, [10,32,45])\
        .build()


In [ ]:
dtc_cv = CrossValidator(estimator=dtc, estimatorParamMaps=grid, evaluator=evaluator, numFolds=5, seed=seed)

In [ ]:
dtc_cv_modelo = dtc_cv.fit(treino)

In [ ]:
dtc_cv_prev = dtc_cv_modelo.transform(teste)

In [ ]:
print(f"Acur?cia: {round(evaluator.evaluate(dtc_cv_prev, {evaluator.metricName: 'accuracy'}), 4)}")
print(f"Precis?o: {round(evaluator.evaluate(dtc_cv_prev, {evaluator.metricName: 'precisionByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"Recall: {round(evaluator.evaluate(dtc_cv_prev, {evaluator.metricName: 'recallByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"F1: {round(evaluator.evaluate(dtc_cv_prev,  {evaluator.metricName: 'fMeasureByLabel', evaluator.metricLabel: 1}), 4)}")

In [ ]:
grid = ParamGridBuilder()\
        .addGrid(rfc.maxDepth,[2,5,10])\
        .addGrid(rfc.maxBins, [10,32,45])\
        .addGrid(rfc.numTrees, [10,20,50])\
        .build()


In [ ]:
rfc_cv = CrossValidator(estimator=rfc, estimatorParamMaps=grid, evaluator=evaluator, numFolds=5, seed=seed)

In [ ]:
rfc_cv_treino = rfc_cv.fit(teste)

In [ ]:
rfc_cv_teste = rfc_cv_treino.transform(teste)

In [ ]:
print(f"Acur?cia: {round(evaluator.evaluate(rfc_cv_teste, {evaluator.metricName: 'accuracy'}), 4)}")
print(f"Precis?o: {round(evaluator.evaluate(rfc_cv_teste, {evaluator.metricName: 'precisionByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"Recall: {round(evaluator.evaluate(rfc_cv_teste, {evaluator.metricName: 'recallByLabel', evaluator.metricLabel: 1}), 4)}")
print(f"F1: {round(evaluator.evaluate(rfc_cv_teste,  {evaluator.metricName: 'fMeasureByLabel', evaluator.metricLabel: 1}), 4)}")

In [ ]:
rfc_cv_treino.bestModel

In [ ]:
best = rfc_cv_treino.bestModel

In [ ]:
rfc_tunning = RandomForestClassifier(numTrees=50,seed=seed)